In [0]:
%pip install databricks-vectorsearch==0.60 -q

## D. Search Methods
- Full-text: retrieve chunks based on exact keyword matches.
- Query: use embeddings to find semantically similar chunks.
- Hybrid: combine the 2 methods above for better relevance.

In [0]:
from databricks.vector_search.client import VectorSearchClient


catalog = "workspace"

# Embedding table (index is a special table)
schema = "feature_model"
index = "docs_chunked_index"
index_name = f"{catalog}.{schema}.{index}"

# init the client
vsc = VectorSearchClient(disable_notice=True)

# get the vector search index for performing searches
index = vsc.get_index(index_name=index_name)
print(index.describe())

### D1. Similarity Search

In [0]:
query_text = "what are some common risks by LLM-generated contents?"
results = index.similarity_search(
    query_text=query_text,
    columns=["path","chunk"],
    num_results=3
)
display(results)

### D2. Hybrid Search
Use this when you want to focus on keywords based on query, besides context.

In [0]:
query_text = "what do BLEU and ROUGE metrics measure?"
results_hybrid = index.similarity_search(
    query_text=query_text,
    columns=["path","chunk"],
    query_type="hybrid",
    num_results=5
)
display(results_hybrid)

### D3. Full-text search
Search based on Keywords only

In [0]:
query_text = "Singapore Academy	of Law"
results_fulltext = index.similarity_search(
    query_text=query_text,
    columns=["path","chunk"],
    query_type="full_text",
    num_results=5
)
display(results_fulltext)

### D4. Filter
- Reference: https://docs.databricks.com/aws/en/vector-search/query-vector-search#use-filters-on-queries
- A query can define filters based on any column in the Delta table. similarity_search returns only rows that match the specified filters
    - by column "path": results for specific document files
    - 

In [0]:
query_text = "Singapore Academy	of Law"
results_filter = index.similarity_search(
    query_text=query_text,
    columns=["path","chunk"],
    filters={
        "path LIKE":"dbfs:/Volumes/workspace/bronze/rag/input/large-language-model-starter-kit.pdf"
    },
    query_type="full_text",
    num_results=3
)
display(results_filter)

## E. Re-ranking to improve Precision
- Embeddings can return results close in meaning but weak in context. Re-ranking help boost precision by re-evaluate the top results using a context-aware model or additional signals.
- In Databricks, we can use the built-in reranker to re-score and re-order top N results from a similarity search, using a deeper contextual understanding.

In [0]:
from databricks.vector_search.reranker import DatabricksReranker

query_text = "what are some common risks by LLM-generated contents?"
results = index.similarity_search(
    query_text=query_text,
    columns=["path","chunk"],
    num_results=3,
    reranker=DatabricksReranker(columns_to_rerank=["chunk"])
)
display(results)